# Week 2-1: 신뢰성 있는 clean table 만들기

이 실습의 핵심은 ETL의 `T`, 그중에서도 **저장 전에 데이터를 믿을 만한 상태로 만드는 기본 변환과 품질 점검**입니다.

최종 목표는 raw 원본도 아니고 보고용 피벗표도 아닌, DB 적재 직전의 중간 정제 테이블을 만드는 것입니다.


## 실습 1의 핵심 목표

- 바로 분석/적재하기엔 불안한 raw 데이터를 본다
- 어떤 컬럼을 남길지 먼저 판단한다
- 자료형, 결측치, 중복, 이상치를 점검한다
- 최소한의 표준화와 정제를 거친다
- **신뢰성 있는 중간 정제 테이블**을 만든다


## 1. 준비

In [19]:
from pathlib import Path

import pandas as pd


데이터 폴더와 입출력 파일 경로를 설정합니다.

In [20]:
DATA_DIR_CANDIDATES = [Path("../data"), Path("week2/data"), Path("data")]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError("data 폴더를 찾을 수 없습니다. download_dataset.py를 먼저 실행하세요.")

RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = DATA_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = RAW_DIR / "movies_metadata.csv"
OUTPUT_PATH = OUTPUT_DIR / "movies_clean_for_load.csv"

DATA_DIR


PosixPath('../data')

## 2. Extract: raw 데이터 읽기

먼저 원본을 그대로 읽습니다. 이 단계에서는 데이터를 고치지 않고, 어떤 문제가 있을 수 있는지 관찰합니다.


In [21]:
movies_raw = pd.read_csv(RAW_PATH, low_memory=False)

print("raw shape:", movies_raw.shape)
movies_raw.head(3)


raw shape: (45466, 24)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0


## 3. raw 데이터의 불안한 지점 보기

CSV는 읽혔다고 바로 저장/분석 가능한 상태가 아닐 수 있습니다. 우선 컬럼, 자료형, 결측치를 가볍게 확인합니다.


In [22]:
check_cols = [
    "id",
    "title",
    "release_date",
    "original_language",
    "runtime",
    "budget",
    "revenue",
    "popularity",
    "vote_average",
    "vote_count",
]

pd.DataFrame({
    "dtype": movies_raw[check_cols].dtypes,
    "missing_count": movies_raw[check_cols].isna().sum(),
    "missing_ratio": (movies_raw[check_cols].isna().mean() * 100).round(2),
})


,dtype,missing_count,missing_ratio
id,str,0,0.00
title,str,6,0.01
release_date,str,87,0.19
original_language,str,11,0.02
runtime,float64,263,0.58
budget,str,0,0.00
revenue,float64,6,0.01
popularity,str,5,0.01
vote_average,float64,6,0.01
vote_count,float64,6,0.01


## 4. Transform 1: 필요한 컬럼만 남기기

모든 컬럼을 다 가져가는 것이 아니라, 저장 목적과 이후 분석에 필요한 컬럼만 고릅니다. 이것도 중요한 Transform 판단입니다.


In [23]:
selected_cols = [
    "id",
    "title",
    "release_date",
    "original_language",
    "runtime",
    "budget",
    "revenue",
    "popularity",
    "vote_average",
    "vote_count",
]

movies = movies_raw[selected_cols].copy()

print("selected shape:", movies.shape)
movies.head(3)


selected shape: (45466, 10)


,id,title,release_date,original_language,runtime,budget,revenue,popularity,vote_average,vote_count
0,862,Toy Story,1995-10-30,en,81.0,30000000,373554033.0,21.946943,7.7,5415.0
1,8844,Jumanji,1995-12-15,en,104.0,65000000,262797249.0,17.015539,6.9,2413.0
2,15602,Grumpier Old Men,1995-12-22,en,101.0,0,0.0,11.7129,6.5,92.0


이번 실습에서 제외한 컬럼도 같이 확인합니다. 제외한다는 것은 쓸모없다는 뜻이 아니라, **이번 clean table의 목적에는 우선순위가 낮다**는 뜻입니다.

예를 들어 `genres`, `production_companies`, `spoken_languages` 같은 컬럼은 문자열 안에 리스트/딕셔너리 구조가 들어 있어서 별도 파싱 실습에 더 적합합니다. `homepage`, `poster_path`, `overview`, `tagline`은 분석용 핵심 지표라기보다 설명/부가 정보에 가깝습니다.


In [24]:
excluded_examples = pd.DataFrame([
    {
        "excluded_column": "genres",
        "reason": "리스트/딕셔너리 형태의 문자열이라 별도 파싱이 필요함",
        "example_value": movies_raw.loc[0, "genres"],
    },
    {
        "excluded_column": "production_companies",
        "reason": "제작사 목록이 중첩 구조로 들어 있어 기본 정제 실습 범위를 넘음",
        "example_value": movies_raw.loc[0, "production_companies"],
    },
    {
        "excluded_column": "overview",
        "reason": "긴 설명 텍스트라 품질 점검용 숫자/날짜 테이블에서는 우선 제외",
        "example_value": movies_raw.loc[0, "overview"],
    },
    {
        "excluded_column": "homepage",
        "reason": "URL 부가 정보이며 결측이 많아 기본 clean table에서는 제외",
        "example_value": movies_raw.loc[0, "homepage"],
    },
    {
        "excluded_column": "poster_path",
        "reason": "이미지 경로 정보라 DB 적재 전 품질 점검 실습의 핵심 컬럼은 아님",
        "example_value": movies_raw.loc[0, "poster_path"],
    },
])

excluded_examples


,excluded_column,reason,example_value
0,genres,리스트/딕셔너리 형태의 문자열이라 별도 파싱이 필요함,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '..."
1,production_companies,제작사 목록이 중첩 구조로 들어 있어 기본 정제 실습 범위를 넘음,"[{'name': 'Pixar Animation Studios', 'id': 3}]"
2,overview,긴 설명 텍스트라 품질 점검용 숫자/날짜 테이블에서는 우선 제외,"Led by Woody, Andy's toys live happily in his ..."
3,homepage,URL 부가 정보이며 결측이 많아 기본 clean table에서는 제외,http://toystory.disney.com/toy-story
4,poster_path,이미지 경로 정보라 DB 적재 전 품질 점검 실습의 핵심 컬럼은 아님,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg


## 5. Transform 2: 컬럼명 최소 표준화

DB에 저장할 때 `id`라는 이름은 다른 테이블의 ID와 헷갈릴 수 있습니다. 여기서는 영화 ID라는 의미가 드러나도록 `movie_id`로 바꿉니다.

컬럼명 표준화는 복잡한 기술이 아니라, 이후 단계에서 같은 의미를 같은 이름으로 부르기 위한 약속입니다.


In [25]:
rename_map = {
    "id": "movie_id",
}

movies = movies.rename(columns=rename_map)
movies.head(3)


,movie_id,title,release_date,original_language,runtime,budget,revenue,popularity,vote_average,vote_count
0,862,Toy Story,1995-10-30,en,81.0,30000000,373554033.0,21.946943,7.7,5415.0
1,8844,Jumanji,1995-12-15,en,104.0,65000000,262797249.0,17.015539,6.9,2413.0
2,15602,Grumpier Old Men,1995-12-22,en,101.0,0,0.0,11.7129,6.5,92.0


## 6. Transform 3: 자료형 변환

문자열로 들어온 숫자/날짜는 계산과 검증이 어렵습니다. 저장 전에 날짜와 숫자 컬럼을 쓸 수 있는 타입으로 바꿉니다.


In [26]:
movies["movie_id"] = pd.to_numeric(movies["movie_id"], errors="coerce")
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")

numeric_cols = ["runtime", "budget", "revenue", "popularity", "vote_average", "vote_count"]
for column in numeric_cols:
    movies[column] = pd.to_numeric(movies[column], errors="coerce")

movies.dtypes


movie_id                    float64
title                           str
release_date         datetime64[us]
original_language               str
runtime                     float64
budget                      float64
revenue                     float64
popularity                  float64
vote_average                float64
vote_count                  float64
dtype: object

## 7. Transform 4: 파생 컬럼 만들기

원본에는 `release_year`가 없지만, 연도별 분석과 검증에 자주 쓰이므로 `release_date`에서 만듭니다.


In [27]:
movies["release_year"] = movies["release_date"].dt.year
movies[["title", "release_date", "release_year"]].head(5)


,title,release_date,release_year
0,Toy Story,1995-10-30,1995.0
1,Jumanji,1995-12-15,1995.0
2,Grumpier Old Men,1995-12-22,1995.0
3,Waiting to Exhale,1995-12-22,1995.0
4,Father of the Bride Part II,1995-02-10,1995.0


## 8. 품질 점검 1: 결측치

결측치는 무조건 나쁜 것이 아니라, 컬럼의 성격에 따라 처리 방식이 달라집니다. 저장 전에 어떤 컬럼에 결측이 있는지 먼저 봅니다.


In [28]:
missing_summary = pd.DataFrame({
    "missing_count": movies.isna().sum(),
    "missing_ratio": (movies.isna().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)

missing_summary


,missing_count,missing_ratio
runtime,263,0.58
release_date,90,0.20
release_year,90,0.20
original_language,11,0.02
title,6,0.01
revenue,6,0.01
popularity,6,0.01
vote_average,6,0.01
vote_count,6,0.01
movie_id,3,0.01


## 9. 결측치 처리

기본 실습에서는 다음 기준을 적용합니다.

- `movie_id`, `title`, `release_date`가 없으면 저장용 영화 테이블로 쓰기 어렵다고 보고 제거
- `original_language`가 없으면 `unknown`으로 대체
- `runtime`은 품질 점검과 이상치 판단에 중요하므로 결측 행 제거


In [29]:
before_rows = len(movies)

movies = movies.dropna(subset=["movie_id", "title", "release_date", "runtime"]).copy()
movies["original_language"] = movies["original_language"].fillna("unknown")

print("처리 전 row 수:", before_rows)
print("처리 후 row 수:", len(movies))
movies.isna().sum()


처리 전 row 수: 45466
처리 후 row 수: 45130


movie_id             0
title                0
release_date         0
original_language    0
runtime              0
budget               0
revenue              0
popularity           0
vote_average         0
vote_count           0
release_year         0
dtype: int64

## 10. 품질 점검 2: 중복

DB에 저장할 때 영화 ID는 한 영화를 가리키는 기준 키가 됩니다. 같은 `movie_id`가 여러 번 나오면 먼저 확인해야 합니다.


In [30]:
duplicate_mask = movies["movie_id"].duplicated(keep=False)
duplicate_count = duplicate_mask.sum()

print("movie_id 중복 행 수:", duplicate_count)
movies.loc[duplicate_mask, ["movie_id", "title", "release_date"]].sort_values("movie_id").head(10)


movie_id 중복 행 수: 59


,movie_id,title,release_date
33826,4912.0,Confessions of a Dangerous Mind,2002-12-30
5865,4912.0,Confessions of a Dangerous Mind,2002-12-30
9165,5511.0,Le Samouraï,1967-10-25
7345,5511.0,Le Samouraï,1967-10-25
44821,10991.0,Pokémon: Spell of the Unknown,2000-07-08
4114,10991.0,Pokémon: Spell of the Unknown,2000-07-08
24844,11115.0,Deal,2008-01-29
14012,11115.0,Deal,2008-01-29
44826,12600.0,Pokémon 4Ever: Celebi - Voice of the Forest,2001-07-06
5535,12600.0,Pokémon 4Ever: Celebi - Voice of the Forest,2001-07-06


기본 실습에서는 같은 `movie_id`가 여러 번 나오면 첫 번째 행만 남깁니다.

In [31]:
movies = movies.drop_duplicates(subset=["movie_id"], keep="first").copy()
print("중복 제거 후 row 수:", len(movies))


중복 제거 후 row 수: 45100


## 11. 품질 점검 3: 논리적으로 이상한 값

값이 비어 있지 않아도 논리적으로 이상할 수 있습니다. 예를 들어 상영시간이 0 이하이거나, 평점이 0~10 범위를 벗어나면 저장 전에 확인해야 합니다.


In [32]:
invalid_runtime = movies["runtime"] <= 0
invalid_budget = movies["budget"] < 0
invalid_revenue = movies["revenue"] < 0
invalid_vote_average = (movies["vote_average"] < 0) | (movies["vote_average"] > 10)

invalid_summary = pd.Series({
    "runtime <= 0": invalid_runtime.sum(),
    "budget < 0": invalid_budget.sum(),
    "revenue < 0": invalid_revenue.sum(),
    "vote_average not in 0~10": invalid_vote_average.sum(),
})

invalid_summary


runtime <= 0                1535
budget < 0                     0
revenue < 0                    0
vote_average not in 0~10       0
dtype: int64

기본 실습에서는 논리적으로 이상한 행을 제거해서 저장용 테이블의 위험을 줄입니다.

In [33]:
before_rows = len(movies)

movies = movies[
    ~invalid_runtime
    & ~invalid_budget
    & ~invalid_revenue
    & ~invalid_vote_average
].copy()

print("이상치 처리 전 row 수:", before_rows)
print("이상치 처리 후 row 수:", len(movies))


이상치 처리 전 row 수: 45100
이상치 처리 후 row 수: 43565


## 12. 저장 직전 clean table 만들기

마지막으로 컬럼 순서를 정리하고, DB에 넣기 전 중간 테이블처럼 사용할 `movies_clean`을 만듭니다.


In [34]:
final_cols = [
    "movie_id",
    "title",
    "release_date",
    "release_year",
    "original_language",
    "runtime",
    "budget",
    "revenue",
    "popularity",
    "vote_average",
    "vote_count",
]

movies_clean = movies[final_cols].copy()
movies_clean["movie_id"] = movies_clean["movie_id"].astype("int64")
movies_clean["release_year"] = movies_clean["release_year"].astype("int64")

print("clean table shape:", movies_clean.shape)
movies_clean.head()


clean table shape: (43565, 11)


,movie_id,title,release_date,release_year,original_language,runtime,budget,revenue,popularity,vote_average,vote_count
0,862,Toy Story,1995-10-30,1995,en,81.0,30000000.0,373554033.0,21.946943,7.7,5415.0
1,8844,Jumanji,1995-12-15,1995,en,104.0,65000000.0,262797249.0,17.015539,6.9,2413.0
2,15602,Grumpier Old Men,1995-12-22,1995,en,101.0,0.0,0.0,11.712900,6.5,92.0
3,31357,Waiting to Exhale,1995-12-22,1995,en,127.0,16000000.0,81452156.0,3.859495,6.1,34.0
4,11862,Father of the Bride Part II,1995-02-10,1995,en,106.0,0.0,76578911.0,8.387519,5.7,173.0


## 13. 저장 전 최종 체크

Load 전에 최소한 다음 정도는 확인합니다.


In [35]:
final_checks = pd.Series({
    "row_count": len(movies_clean),
    "duplicate_movie_id": movies_clean["movie_id"].duplicated().sum(),
    "missing_movie_id": movies_clean["movie_id"].isna().sum(),
    "missing_title": movies_clean["title"].isna().sum(),
    "missing_release_date": movies_clean["release_date"].isna().sum(),
    "invalid_runtime": (movies_clean["runtime"] <= 0).sum(),
})

final_checks


row_count               43565
duplicate_movie_id          0
missing_movie_id            0
missing_title               0
missing_release_date        0
invalid_runtime             0
dtype: int64

## 14. Load 전 중간 결과 저장

이 파일은 raw 원본도 아니고 보고용 집계표도 아닙니다. DB에 적재하기 직전의 clean table입니다.


In [36]:
movies_clean.to_csv(OUTPUT_PATH, index=False)
print("저장 완료:", OUTPUT_PATH)


저장 완료: ../data/output/movies_clean_for_load.csv


## 15. 다음 단계: Load로 이어가기

이제 `movies_clean`은 DB 테이블로 적재할 후보가 됩니다. 실제 Load 전에는 테이블명, 키, 컬럼 타입, 적재 후 검증 기준을 정해야 합니다.


In [37]:
load_plan = pd.DataFrame([
    {
        "table_name": "movies_clean",
        "source_dataframe": "movies_clean",
        "grain": "영화 1개당 1행",
        "key_column": "movie_id",
        "example_checks": "row 수, movie_id 중복 여부, 핵심 컬럼 결측 여부",
    }
])

load_plan


,table_name,source_dataframe,grain,key_column,example_checks
0,movies_clean,movies_clean,영화 1개당 1행,movie_id,"row 수, movie_id 중복 여부, 핵심 컬럼 결측 여부"


## 16. 정리

이번 실습에서 만든 결과물은 **DB 적재에 활용할 수 있는 신뢰성 있는 정제 테이블**입니다.

- 직관적인 컬럼만 선택
- 컬럼명 최소 표준화
- 자료형 변환
- 결측치 처리
- 중복 체크와 제거
- 논리적으로 이상한 값 점검과 처리
- `release_year` 파생 컬럼 생성

즉, 실습 1은 사람이 읽는 표나 시각화 결과를 만드는 것이 아니라, 저장 전에 데이터를 믿을 만한 상태로 만드는 기본 변환과 품질 점검을 해보는 실습입니다.
